Section 1: Descriptive Statistics

Amani Insurance claims case study


Deliverable: To Calculate and interpret mean, median, mode, and standard deviation.

In [2]:
import numpy as np
import pandas as pd

FILE_PATH = "insurance_claims_messy.csv"


In [3]:
df = pd.read_csv(FILE_PATH)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(5)

Shape: 509 rows, 9 columns


,claim_id,policy_number,claim_type,claim_amount_kes,claim_date,region,status,assessor,notes
0,CLM-00476,AMI-00079,property fire,1222500,"April 20, 2024",Nakuru,Pending,M. Hassan,site visit done
1,CLM-00108,AMI-00186,health inpatient,"142,600",30.04.2024,Kisumu,rejected,P. Achieng,NaN
2,CLM-00152,AMI-00132,Motor Accident,228000,26-05-2024,NAIROBI,pending,S. Kimani,customer unreachable
3,CLM-00182,AMI-00037,Property-Burglary,243800,2024-02-02,KISUMU,Rejected,M. Hassan,police abstract attached
4,CLM-00197,AMI-00131,motor accident,216800,"March 06, 2024",eldoret,rejected,J. Mwangi,site visit done


In [4]:
df.dtypes

claim_id            str
policy_number       str
claim_type          str
claim_amount_kes    str
claim_date          str
region              str
status              str
assessor            str
notes               str
dtype: object

In [5]:
# .astype(str)          -> make sure every value is treated as text first
# .str.replace(",", "")  -> remove thousands-commas, e.g. "142,600" -> "142600"
# .str.strip()            -> remove stray leading/trailing spaces
df["claim_amount_kes"] = (
    df["claim_amount_kes"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# errors="coerce": anything that still isn't a valid number (blanks, junk)
# becomes NaN instead of raising an error and stopping the whole script.
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce")

print("Dtype after cleaning:", df["claim_amount_kes"].dtype)
print(f"Missing claim_amount_kes values: {df['claim_amount_kes'].isna().sum()}")
print(f"Negative claim_amount_kes values: {(df['claim_amount_kes'] < 0).sum()}")

Dtype after cleaning: float64
Missing claim_amount_kes values: 30
Negative claim_amount_kes values: 7


In [6]:
df.dtypes

claim_id                str
policy_number           str
claim_type              str
claim_amount_kes    float64
claim_date              str
region                  str
status                  str
assessor                str
notes                   str
dtype: object

Mean, median, mode, standard deviation

In [7]:
df.describe()

,claim_amount_kes
count,4.790000e+02
mean,3.824182e+05
std,7.202407e+05
min,-1.325300e+06
25%,6.925000e+04
50%,2.005000e+05
75%,5.890000e+05
max,1.011200e+07


A single set of stats for all claims hides a lot. Motor-accident claims and property-fire claims are not remotely on the same scale — let's split by claim_type and compute mean/median/std/count for each group separately.

In [15]:
# Canonicalise claim_type first, so "motor accident" / "MOTOR-ACCIDENT" /
# "Motor-Accident" all get counted as the SAME group instead of three
# different ones. (Full categorical cleanup is Part 3's job -- this is
# just enough to make groupby() behave sensibly here.)
df["claim_type_clean"] = (
    df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
)

usable = df[df["claim_amount_kes"] > 0]

# groupby(...).agg(...) computes all four statistics per group in one call.
summary = usable.groupby("claim_type_clean")["claim_amount_kes"].agg(
    mean="mean", median="median", std="std", count="count"
).sort_values("mean", ascending=False)

summary.round(0)

,mean,median,std,count
claim_type_clean,,,,
property-fire,1093434.0,939100.0,1375860.0,79
motor-theft,691287.0,617200.0,480506.0,83
property-burglary,262212.0,276700.0,137776.0,73
motor-accident,200822.0,183350.0,168831.0,72
health-inpatient,110861.0,109200.0,64931.0,89
health-outpatient,8386.0,8300.0,4023.0,76
